## Transform Payment Data

1. Extract Data and Time from the payment_timestamp and create new columns payment_data and payment_time
2. Map payment_status to contain descriptive values
   (1-Success, 2-Pending, 3-Cancelled, 4-Failed)
3. Write transformed data to the Silver Schema

In [0]:
df = spark.table('gizmobox.bronze.py_payments')
display(df)

1. Extract Date and Time from payment_timestamp

In [0]:
from pyspark.sql import functions as F

df_extracted_payment = (
                         df
                         .select(
                             'payment_id', 
                             'order_id',
                             F.date_format('payment_timestamp','yyyy-MM-dd').cast('date').alias('payment_date'),
                             F.date_format('payment_timestamp','HH-mm-ss').alias('payment_time'),
                             'payment_status',
                             'payment_method'
                         )

)
display(df_extracted_payments)

2. Map payment_status to contain descriptive values (1-Success, 2-Pending, 3-Cancelled, 4-Failed)

In [0]:
df_mapped_payment = (
                        df_extracted_payment
                        .select (
                            'payment_id',
                            'order_id',
                            'payment_date',
                            'payment_time',
                            F.when(df_extracted_payment.payment_status == 1,'Success')
                             .when(df_extracted_payment.payment_status == 2,'Pending')
                             .when(df_extracted_payment.payment_status == 3,'cancelled')
                             .when(df_extracted_payment.payment_status == 4,'Failed')
                             .alias('payment_status'),
                             'payment_method'
                            )
)
display(df_mapped_payment)

3. Write transformed data to the Silver Schema

In [0]:
df_mapped_payment.writeTo('gizmobox.silver.py_payment').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_payment;